In [1]:
import os, random
from datetime import date, datetime, timedelta
import numpy as np
import pandas as pd
from dateutil.relativedelta import relativedelta
from faker import Faker

SEED = 42
random.seed(SEED); np.random.seed(SEED)
rng = np.random.default_rng(SEED)
Faker.seed(SEED); fake = Faker("en_IN")

START, AS_OF = date(2024, 11, 1), date(2026, 10, 31)
N_MEMBERS = 150
OUT = "data"; os.makedirs(OUT, exist_ok=True)

def dt(d, h=0, m=0):
    return datetime(d.year, d.month, d.day, h, m)

# ---------------- reference tables ----------------
plans = pd.DataFrame(
    [(1, "Monthly", 1, 1499.00), (2, "Quarterly", 3, 3999.00),
     (3, "Half-Yearly", 6, 6999.00), (4, "Annual", 12, 11999.00)],
    columns=["plan_id", "plan_name", "duration_months", "price"])
PLAN_IDS = plans.plan_id.tolist()
PLAN_W = [0.50, 0.25, 0.10, 0.15]
BASE_RENEW = {1: 0.45, 2: 0.60, 3: 0.68, 4: 0.78}
PRICE = dict(zip(plans.plan_id, plans.price))
DURM = dict(zip(plans.plan_id, plans.duration_months))

rooms = pd.DataFrame(
    [(1, "Studio A", 25), (2, "Studio B", 20), (3, "Spin Room", 18),
     (4, "Main Floor", 30), (5, "Boxing Ring", 12)],
    columns=["room_id", "room_name", "capacity"])
ROOM_CAP = dict(zip(rooms.room_id, rooms.capacity))

# id, name, category, duration, popularity, base_show, allowed_rooms
CT = [
    (1, "Yoga", "Mind & Body", 60, 0.90, 0.85, [1, 2]),
    (2, "HIIT", "Cardio", 45, 1.00, 0.75, [4, 1]),
    (3, "Zumba", "Cardio", 60, 0.95, 0.80, [1, 4]),
    (4, "Spin", "Cardio", 45, 1.00, 0.82, [3]),
    (5, "Pilates", "Mind & Body", 50, 0.70, 0.84, [2]),
    (6, "CrossFit", "Strength", 60, 0.80, 0.70, [4]),
    (7, "Strength Training", "Strength", 60, 0.75, 0.78, [4]),
    (8, "Boxing", "Combat", 60, 0.60, 0.72, [5]),
    (9, "Meditation", "Mind & Body", 30, 0.35, 0.88, [2]),
    (10, "Aerobics", "Cardio", 50, 0.65, 0.77, [1, 4]),
]
class_types = pd.DataFrame([c[:4] for c in CT],
                           columns=["class_type_id", "class_name", "category", "duration_min"])
CT_IDS = [c[0] for c in CT]
DUR = {c[0]: c[3] for c in CT}
POP = {c[0]: c[4] for c in CT}
BASE_SHOW = {c[0]: c[5] for c in CT}
ALLOWED = {c[0]: c[6] for c in CT}
CT_W = np.array([POP[i] for i in CT_IDS]); CT_W = CT_W / CT_W.sum()

# trainers
SPEC = {"Yoga & Mindfulness": [1, 5, 9], "Cardio": [2, 3, 4, 10],
        "Strength & Conditioning": [6, 7], "Combat Sports": [8]}
spec_list = (["Yoga & Mindfulness"] * 3 + ["Cardio"] * 4 +
             ["Strength & Conditioning"] * 3 + ["Combat Sports"] * 2)
trainer_rows, TR_FOR = [], {i: [] for i in CT_IDS}
for tid, sp in enumerate(spec_list, start=1):
    name = fake.name()
    trainer_rows.append((tid, name, f"trainer{tid}@fitpulse.example", fake.phone_number(),
                         sp, fake.date_between(date(2018, 1, 1), date(2024, 10, 1))))
    for ct in SPEC[sp]:
        TR_FOR[ct].append(tid)
trainers = pd.DataFrame(trainer_rows,
                        columns=["trainer_id", "full_name", "email", "phone", "specialization", "hire_date"])

# ---------------- members ----------------
MONTH_W = {1: 2.0, 2: 1.3, 3: 1.0, 4: 0.9, 5: 0.9, 6: 0.8,
           7: 0.8, 8: 0.9, 9: 1.1, 10: 1.0, 11: 0.8, 12: 0.9}
signup_days = [START + timedelta(i) for i in range((AS_OF - timedelta(20) - START).days + 1)]
sw = np.array([MONTH_W[d.month] for d in signup_days]); sw /= sw.sum()
first_start = {m: signup_days[i] for m, i in
               zip(range(1, N_MEMBERS + 1), rng.choice(len(signup_days), N_MEMBERS, p=sw))}

eng = {m: float(rng.beta(2, 2.5)) for m in range(1, N_MEMBERS + 1)}   # latent engagement
member_rows = []
for m in range(1, N_MEMBERS + 1):
    fn, ln = fake.first_name(), fake.last_name()
    age = int(np.clip(rng.normal(31, 8), 18, 62))
    dob = AS_OF - timedelta(days=age * 365 + int(rng.integers(0, 365)))
    member_rows.append((
        m, fn, ln, f"{fn}.{ln}{m}@example.com".lower(), fake.phone_number(),
        rng.choice(["Male", "Female", "Other"], p=[0.52, 0.46, 0.02]), dob,
        rng.choice(["Bengaluru", "Mysuru", "Mangaluru", "Hubballi"], p=[0.70, 0.12, 0.10, 0.08]),
        first_start[m] - timedelta(days=int(rng.integers(0, 15)))))
members = pd.DataFrame(member_rows, columns=[
    "member_id", "first_name", "last_name", "email", "phone", "gender",
    "date_of_birth", "city", "registered_on"])

# ---------------- memberships (renewal chains) ----------------
mem_rows, final_churn, mid = [], {}, 0
for m in range(1, N_MEMBERS + 1):
    e, start, prev = eng[m], first_start[m], None
    plan = int(rng.choice(PLAN_IDS, p=PLAN_W))
    while True:
        mid += 1
        end = start + relativedelta(months=DURM[plan]) - timedelta(days=1)
        disc = float(rng.choice([1.0, 0.9, 0.85], p=[0.7, 0.2, 0.1]))
        mem_rows.append(dict(
            membership_id=mid, member_id=m, plan_id=plan, start_date=start, end_date=end,
            price_paid=round(PRICE[plan] * disc, 2),
            status="Expired" if end < AS_OF else "Active", renewed_from_id=prev))
        if end >= AS_OF:
            final_churn[mid] = False; break
        p = float(np.clip(BASE_RENEW[plan] + 0.5 * (e - 0.4), 0.05, 0.95))
        if rng.random() >= p:
            final_churn[mid] = True; break
        final_churn[mid] = False
        prev = mid
        start = min(end + timedelta(days=1 + int(rng.integers(0, 6))), AS_OF)
        if rng.random() > 0.65:                      # 35% switch plan on renewal
            plan = int(rng.choice(PLAN_IDS, p=PLAN_W))
memberships = pd.DataFrame(mem_rows)
memberships["renewed_from_id"] = memberships["renewed_from_id"].astype("Int64")

def in_last_28(r, d):
    return final_churn[r["membership_id"]] and (r["end_date"] - d).days <= 28

# ---------------- classes (no room/trainer overlap) ----------------
WD_H, WD_W = [6, 7, 8, 12, 17, 18, 19, 20], np.array([.16, .16, .06, .05, .14, .20, .15, .08])
WE_H, WE_W = [8, 9, 10, 11, 17], np.array([.25, .30, .25, .10, .10])
PER_DAY = {0: 2, 1: 2, 2: 2, 3: 2, 4: 2, 5: 2, 6: 1}
room_busy, tr_busy = {}, {}

def is_free(busy, key, s, e):
    return all(e <= bs or s >= be for bs, be in busy.get(key, []))

class_rows = []
d = START
while d <= AS_OF:
    wd = d.weekday()
    H, W = (WD_H, WD_W) if wd < 5 else (WE_H, WE_W)
    for _ in range(PER_DAY[wd]):
        for _try in range(30):
            ct = int(rng.choice(CT_IDS, p=CT_W))
            room = int(rng.choice(ALLOWED[ct])); tr = int(rng.choice(TR_FOR[ct]))
            s = dt(d, int(rng.choice(H, p=W / W.sum()))); e_ = s + timedelta(minutes=DUR[ct])
            if is_free(room_busy, room, s, e_) and is_free(tr_busy, tr, s, e_):
                room_busy.setdefault(room, []).append((s, e_))
                tr_busy.setdefault(tr, []).append((s, e_))
                class_rows.append((ct, tr, room, s, e_)); break
    d += timedelta(days=1)
classes = pd.DataFrame(class_rows, columns=["class_type_id", "trainer_id", "room_id", "start_ts", "end_ts"])
classes = classes.sort_values("start_ts").reset_index(drop=True)
classes.insert(0, "class_id", classes.index + 1)

# ---------------- bookings + class attendance ----------------
active_cache, member_busy = {}, {}
bk_rows, att_rows, bid = [], [], 0
for c in classes.itertuples(index=False):
    day = c.start_ts.date()
    if day not in active_cache:
        active_cache[day] = [r for r in mem_rows if r["start_date"] <= day <= r["end_date"]]
    active = active_cache[day]
    if not active:
        continue
    cap = ROOM_CAP[c.room_id]
    hf = 1.3 if c.start_ts.hour in (6, 7, 18, 19) else 0.85
    k = min(int(rng.poisson(cap * POP[c.class_type_id] * hf * 0.9)), cap, len(active))
    if k == 0:
        continue
    w = np.array([eng[r["member_id"]] ** 1.5 + 0.02 for r in active]); w /= w.sum()
    for idx in rng.choice(len(active), size=k, replace=False, p=w):
        r = active[idx]; m = r["member_id"]; e = eng[m]
        if not is_free(member_busy, m, c.start_ts, c.end_ts):
            continue
        member_busy.setdefault(m, []).append((c.start_ts, c.end_ts))
        bid += 1
        booked_at = max(c.start_ts - timedelta(hours=float(rng.uniform(2, 72))), dt(r["start_date"]))
        cancelled = rng.random() < 0.08 + 0.25 * (1 - e)
        bk_rows.append((bid, m, c.class_id, booked_at, "Cancelled" if cancelled else "Confirmed"))
        if cancelled:
            continue
        p = BASE_SHOW[c.class_type_id] * (0.7 + 0.5 * e)
        if c.start_ts.hour <= 6: p *= 0.92
        if in_last_28(r, day): p *= 0.6
        if rng.random() < min(p, 0.97):
            att_rows.append((m, bid,
                             c.start_ts - timedelta(minutes=int(rng.integers(0, 13))),
                             c.end_ts + timedelta(minutes=int(rng.integers(0, 11)))))
bookings = pd.DataFrame(bk_rows, columns=["booking_id", "member_id", "class_id", "booked_at", "status"])

# ---------------- gym-floor visits (walk-ins) ----------------
DOW = [1.25, 1.10, 1.10, 1.00, 0.90, 0.95, 0.70]
HOURS = list(range(5, 23))
HW = np.array([.03, .09, .11, .07, .05, .04, .03, .04, .04, .03, .03, .05, .09, .12, .09, .05, .03, .01])
HW = HW / HW.sum()
for r in mem_rows:
    m, e = r["member_id"], eng[r["member_id"]]
    last = min(r["end_date"], AS_OF)
    for i in range((last - r["start_date"]).days + 1):
        day = r["start_date"] + timedelta(i)
        lam = 0.4 + 4.6 * e
        if i < 21: lam *= 1.3
        if in_last_28(r, day): lam *= 0.35
        if rng.random() < min(0.95, lam / 7 * DOW[day.weekday()]):
            ci = dt(day, int(rng.choice(HOURS, p=HW)), int(rng.integers(0, 60)))
            co = ci + timedelta(minutes=int(np.clip(rng.normal(75, 20), 25, 150)))
            att_rows.append((m, None, ci, co))

attendance = pd.DataFrame(att_rows, columns=["member_id", "booking_id", "check_in_ts", "check_out_ts"])
attendance = attendance.sort_values("check_in_ts").reset_index(drop=True)
attendance.insert(0, "attendance_id", attendance.index + 1)
attendance["booking_id"] = attendance["booking_id"].astype("Int64")

# ---------------- integrity checks ----------------
conf = bookings[bookings.status == "Confirmed"].merge(classes, on="class_id").merge(rooms, on="room_id")
assert (conf.groupby("class_id").size().reindex(conf.class_id.unique())
        <= conf.drop_duplicates("class_id").set_index("class_id").capacity).all()
assert not bookings.duplicated(["member_id", "class_id"]).any()
assert attendance.booking_id.dropna().is_unique

# ---------------- export ----------------
tables = dict(plans=plans, members=members, memberships=memberships, trainers=trainers,
              class_types=class_types, rooms=rooms, classes=classes,
              bookings=bookings, attendance=attendance)
for name, df in tables.items():
    df.to_csv(f"{OUT}/{name}.csv", index=False)
    print(f"{name:12s} {len(df):>7,d} rows")

# ---------------- sanity summary ----------------
exp = memberships[memberships.status == "Expired"].copy()
exp["renewed"] = exp.membership_id.isin(set(memberships.renewed_from_id.dropna().astype(int)))
print("\nRenewal rate by plan:\n",
      exp.groupby("plan_id").renewed.mean().round(2).rename(index=dict(zip(plans.plan_id, plans.plan_name))))
print("\nMean class fill rate:",
      round((conf.groupby("class_id").size() / conf.drop_duplicates("class_id").set_index("class_id").capacity).mean(), 2))
print("Class no-show rate:",
      round(1 - attendance.booking_id.notna().sum() / (bookings.status == "Confirmed").sum(), 2))

plans              4 rows
members          150 rows
memberships      314 rows
trainers          12 rows
class_types       10 rows
rooms              5 rows
classes        1,356 rows
bookings      22,107 rows
attendance    23,159 rows

Renewal rate by plan:
 plan_id
Monthly        0.51
Quarterly      0.69
Half-Yearly    0.75
Annual         0.93
Name: renewed, dtype: float64

Mean class fill rate: 0.56
Class no-show rate: 0.27
